# **Process:**                      
Cell 1  → defines all config variables (CARDIOLOGY_ROOTS, EXCLUDED_SUBTREES, etc.)                                                                             
Cell 2  → BFS on IS_A to build cardiology_ids + external_ids

Cell 3  → applies exclusions, builds all_ids

Cell 4  → loads Concept file, filters to all_ids

Cell 5  → loads Relationship file, filters + applies TYPE_MAP

Cell 6  → loads Description file

Cell 7  → loads Language refset (preferred terms)

Cell 8  →

Cell 9  → builds final dataframes

Cell 10 → writes TSV files to OUTPUT_DIR

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import csv
import json
from datetime import datetime
from collections import Counter

import pandas as pd

In [ ]:
# ===================================================================
# SNOMED CT Cardiology Extraction — Cell 1 Configuration
# Scope  : Symptom → Diagnosis workflow
# Version: v5.0  |  Date: 2026-05-16
# ===================================================================

CURATION_VERSION = "v5.0"
CURATION_DATE    = "2026-05-16"

CURATION_NOTES = """
Cardiology-focused SNOMED CT extraction.

Scope: Symptom → Diagnosis workflow only.
  IN  : Symptoms, vital-sign findings, bedside/lab diagnostics,
        diagnostic ECG/echo/CXR findings, ACS/HF/arrhythmia diseases.
  OUT : Post-diagnosis therapeutic procedures (PCI, ablation, valve
        surgery, pacemakers, ICD, transplant, rehab), non-cardiac
        anatomy, dermatological/retinal/obstetric contamination,
        hereditary neurological syndromes with no cardiac relevance.

All exclusions validated through IS-A hierarchy tracing in the
SNOMED CT browser (browser.ihtsdotools.org) or confirmed during
v1–v3 extraction audits.
"""


In [ ]:
# ===================================================================
# PATHS & CONSTANTS
# ===================================================================

SNAPSHOT_PATH = "/content/drive/MyDrive/PFE/data_raw/Snapshot"
OUTPUT_DIR    = "/content/drive/MyDrive/PFE/MedGraph"
IS_A_TYPE     = "116680003"
BATCH_SIZE    = 50_000

csv.field_size_limit(2147483647)

131072

In [ ]:
# ===================================================================
# PRIMARY ROOTS — BFS starts from each of these SCTIDs
# ===================================================================

DISEASE_ROOTS = {
    "49601007": "Cardiology",   # Disorder of cardiovascular system
}

SYMPTOM_ROOTS = {
    "29857009":  "Cardiology",  # Chest pain (finding)
    "230145002": "Cardiology",  # Dyspnea (finding)
    "80313002":  "Cardiology",  # Palpitations (finding)
    "271594007": "Cardiology",  # Syncope (finding)
    "52613005":  "Cardiology",  # Diaphoresis (finding)
}

PROCEDURE_ROOTS = {
    "29303009":   "Cardiology", # Electrocardiographic procedure
    "40701008":   "Cardiology", # Echocardiography
    "1290970007": "Cardiology", # Plain X-ray of heart (CXR)
}

FINDING_ROOTS = {
    "301120008": "Cardiology",  # Finding present on electrocardiogram
                                # → ST-elevation, LBBB, Q-waves, etc.
}

LAB_ROOTS = {
    "105000003": "Cardiology",  # Troponin measurement
    "390917008": "Cardiology",  # BNP measurement
    "25607008":  "Cardiology",  # D-dimer measurement
}

# Merge all roots into a single flat dict — used by Cell 2 BFS
CARDIOLOGY_ROOTS = {
    **DISEASE_ROOTS,
    **SYMPTOM_ROOTS,
    **PROCEDURE_ROOTS,
    **FINDING_ROOTS,
    **LAB_ROOTS,
}

In [ ]:
# ===================================================================
# PROTECTED SEEDS
# Concepts that MUST survive even if their IS-A ancestor is excluded.
# Cell 2 reads this as {sctid: type_string} — keep values as strings.
# ===================================================================

PROTECTED_SEEDS = {
    "16652001": "Cardiology",   # Fabry disease → causes hypertrophic CMP
    "31848007": "Cardiology",   # CREST syndrome → pulmonary arterial HTN
    "13645005": "External",     # COPD → cor pulmonale / pulmonary HTN
    "398254007": "External",    # Pre-eclampsia → major CV risk condition
    "401314000": "Cardiology",  # NSTEMI — critical ACS subtype
    "225566008": "Cardiology",  # Ischaemic chest pain — core cardiac symptom
}


In [ ]:
# ===================================================================
# EXCLUDED TOP-LEVEL SNOMED HIERARCHIES
# These 19-root branches are unwanted in their entirety.
# ===================================================================

EXCLUDED_ROOTS = {
    "410607006",          # Organism (bacteria, viruses, parasites)
    "308916002",          # Environment or geographical location
    "260787004",          # Physical object (removes ECG hardware nodes)
    "419891008",          # Record artifact
    "78621006",           # Physical force
    "370115009",          # Special concept
    "900000000000441003", # SNOMED CT Model Component
}

In [ ]:
# ===================================================================
# EXCLUSION GROUPS
# Each group is a dict: { sctid: {"label": ..., "reason": ...} }
# The BFS in Cell 2 will exclude each SCTID AND all its descendants.
# Rule: always exclude a BRANCH, never a single leaf alone.
# ===================================================================

# -------------------------------------------------------------------
# GROUP 1 — Therapeutic / post-diagnosis cardiac procedures
# These occur AFTER diagnosis is confirmed — out of scope.
# -------------------------------------------------------------------
THERAPEUTIC_PROCEDURES = {
    "415070008": {
        "label":  "Percutaneous coronary intervention (PCI)",
        "reason": "Post-diagnosis therapeutic procedure",
    },
    "232717009": {
        "label":  "Coronary artery bypass grafting (CABG)",
        "reason": "Post-diagnosis therapeutic procedure",
    },
    "18286008": {
        "label":  "Catheter ablation of cardiac tissue",
        "reason": "Post-diagnosis therapeutic procedure (ablation, cryo, PVI)",
    },
    "233174007": {
        "label":  "Cardiac pacemaker implantation",
        "reason": "Post-diagnosis device therapy",
    },
    "395218007": {
        "label":  "Implantation of internal cardiac defibrillator (ICD)",
        "reason": "Post-diagnosis device therapy",
    },
    "64915003": {
        "label":  "Operation on heart valve",
        "reason": "Post-diagnosis surgical therapy (covers repair + replacement + TAVI)",
    },
    "870255009": {
        "label":  "Cardiac resynchronisation therapy (CRT)",
        "reason": "Post-diagnosis device therapy",
    },
    "32413006": {
        "label":  "Transplantation of heart",
        "reason": "Post-diagnosis surgical therapy",
    },
    "232965003": {
        "label":  "Implantation of ventricular assist device (VAD)",
        "reason": "Post-diagnosis device therapy",
    },
    "704050007": {
        "label":  "Cardiac rehabilitation programme",
        "reason": "Long-term chronic management — outside diagnostic scope",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 2 — Non-cardiology anatomy
# Body structure concepts outside cardiac/vascular anatomy.
# -------------------------------------------------------------------
NON_CARDIO_ANATOMY = {
    "81745001": {
        "label":  "Eye structure",
        "reason": "Ophthalmology anatomy — no cardiology relevance",
    },
    "39937001": {
        "label":  "Skin structure",
        "reason": "Dermatology anatomy",
    },
    "127882003": {
        "label":  "Female genital organ structure",
        "reason": "Gynaecology anatomy",
    },
    "51289009": {
        "label":  "Digestive tract structure",
        "reason": "Gastroenterology anatomy",
    },
    "122489005": {
        "label":  "Urinary system structure",
        "reason": "Nephrology/urology anatomy",
    },
}


In [ ]:
# -------------------------------------------------------------------
# GROUP 3 — Dermatology / vascular-of-skin contamination
# These enter the graph through SNOMED vascular inheritance.
# -------------------------------------------------------------------
DERMATOLOGY_SKIN = {
    "11263005": {
        "label":  "Vascular disease of skin",
        "reason": "Branch root for strawberry nevus family; dermatology",
    },
    "83343001": {
        "label":  "Capillary hemangioma (morphologic abnormality)",
        "reason": "Dermatology contamination via vascular morphology",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 4 — Retinal / diabetic eye complications
# These appear because diabetic retinopathy inherits through
# 'vascular disorder' hierarchy nodes shared with cardiology.
# -------------------------------------------------------------------
RETINAL_DIABETIC = {
    "25412000": {
        "label":  "Microaneurysm of retinal artery due to diabetes",
        "reason": "Ophthalmology — diabetic retinopathy branch",
    },
    "314014002": {
        "label":  "Ischaemic maculopathy due to diabetes",
        "reason": "Ophthalmology — diabetic retinopathy branch",
    },
    "314015001": {
        "label":  "Mixed maculopathy due to diabetes",
        "reason": "Ophthalmology — diabetic retinopathy branch",
    },
    "399866003": {
        "label":  "Venous beading of retina due to diabetes",
        "reason": "Ophthalmology — diabetic retinopathy branch",
    },
    "399868002": {
        "label":  "Intraretinal microvascular anomaly due to diabetes",
        "reason": "Ophthalmology — diabetic retinopathy branch",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 5 — CKD × diabetes hypertension cross-products
# SNOMED generates very specific combined-condition concepts.
# Those relevant to cardiology are already captured via the HTN
# hierarchy under 49601007; these are the non-cardiac remnants.
# -------------------------------------------------------------------
CKD_DIABETES_CROSS = {
    "140111000119107": {
        "label":  "HTN in CKD stage 4 due to type 2 diabetes",
        "reason": "CKD/nephrology cross-product — cardiac component captured elsewhere",
    },
    "1332436006": {
        "label":  "HTN in CKD stage 3B due to type 1 diabetes",
        "reason": "CKD/nephrology cross-product",
    },
    "1332441003": {
        "label":  "HTN in CKD stage 3A due to type 1 diabetes",
        "reason": "CKD/nephrology cross-product",
    },
    "1332442005": {
        "label":  "HTN in CKD stage 3 due to type 1 diabetes",
        "reason": "CKD/nephrology cross-product",
    },
    "1332464001": {
        "label":  "HTN in CKD stage 2 due to type 1 diabetes",
        "reason": "CKD/nephrology cross-product",
    },
    "1332467008": {
        "label":  "HTN in CKD stage 5 due to type 1 diabetes",
        "reason": "CKD/nephrology cross-product",
    },
}


In [ ]:
# -------------------------------------------------------------------
# GROUP 6 — Obstetric / nutritional contamination
# -------------------------------------------------------------------
OBSTETRIC_NUTRITIONAL = {
    "76751001": {
        "label":  "Diabetes mellitus in mother complicating pregnancy",
        "reason": "Obstetric scope — not a cardiac presentation",
    },
    "55565007": {
        "label":  "Heart failure following obstetric procedure",
        "reason": "Obstetric complication — captured generically via HF hierarchy",
    },
    "1264255000": {
        "label":  "Complete induced termination of pregnancy",
        "reason": "Obstetric procedure — no cardiac diagnostic relevance",
    },
    "75524006": {
        "label":  "Malnutrition-related diabetes mellitus",
        "reason": "Nutritional/endocrine — not a cardiac presentation",
    },
    "276792008": {
        "label":  "Pulmonary hypertension with extreme obesity",
        "reason": "Obesity-specific pulmonary combination outside cardiac triage",
    },
    "190966007": {
        "label":  "Extreme obesity with alveolar hypoventilation",
        "reason": "Obesity-specific pulmonary combination outside cardiac triage",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 7 — Root-level cardiovascular clean cuts
# Branches confirmed during v3 audit to have no triage value.
# -------------------------------------------------------------------
ROOT_LEVEL_CUTS = {
    "459157005": {
        "label":  "Abscess of cardiovascular heterograft",
        "reason": "Surgical complication concept — not a diagnostic entity",
    },
    "459156001": {
        "label":  "Abscess of cardiovascular homograft",
        "reason": "Surgical complication concept",
    },
    "459155002": {
        "label":  "Abscess of cardiovascular structure of trunk",
        "reason": "Surgical complication concept",
    },
    "461430004": {
        "label":  "Abscess of vascular cardiac conduit",
        "reason": "Surgical complication concept",
    },
    "397893009": {
        "label":  "Cardiovascular morbidity",
        "reason": "Statistical/epidemiological concept — not a clinical finding",
    },
    "62914000": {
        "label":  "Cerebrovascular disease",
        "reason": "Neurology scope — no cardiac diagnostic relevance",
    },
    "432995009": {
        "label":  "Fetal cardiovascular disorder",
        "reason": "Fetal/neonatal medicine scope",
    },
    "286561001": {
        "label":  "Foreign body of cardiovascular structure",
        "reason": "Surgical/trauma concept",
    },
    "363696006": {
        "label":  "Neonatal cardiovascular disorder",
        "reason": "Neonatal medicine scope",
    },
    "721573003": {
        "label":  "Neoplasm of cardiovascular system",
        "reason": "Oncology scope",
    },
    "276510003": {
        "label":  "Perinatal cardiovascular disorder",
        "reason": "Obstetric/neonatal medicine scope",
    },
    "472814001": {
        "label":  "Spontaneous closure of fenestration of atrial tunnel",
        "reason": "Congenital anatomy variant — not a diagnostic finding",
    },
    "472813007": {
        "label":  "Spontaneous closure of fenestration of interatrial septum",
        "reason": "Congenital anatomy variant — not a diagnostic finding",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 8 — Hereditary cerebrovascular / neurological syndromes
# These enter via vascular hierarchy but have no cardiac triage role.
# -------------------------------------------------------------------
HEREDITARY_CEREBROVASCULAR = {
    "771476007": {
        "label":  "Autosomal recessive leukoencephalopathy with ischaemic stroke and retinitis pigmentosa",
        "reason": "Neurological hereditary syndrome",
    },
    "1197429000": {
        "label":  "Cathepsin A-related arteriopathy, strokes and leukoencephalopathy",
        "reason": "Neurological hereditary syndrome",
    },
    "703219008": {
        "label":  "CARASIL (cerebral autosomal recessive arteriopathy with subcortical infarcts)",
        "reason": "Neurological hereditary syndrome",
    },
    "778060000": {
        "label":  "COL4A1 familial vascular leukoencephalopathy",
        "reason": "Neurological hereditary syndrome",
    },
    "95656000": {
        "label":  "Familial hemiplegic migraine (parent)",
        "reason": "Neurological syndrome — enters via vascular hierarchy",
    },
    "1260329005": {
        "label":  "Familial hemiplegic migraine type 1",
        "reason": "Neurological syndrome",
    },
    "1260330000": {
        "label":  "Familial hemiplegic migraine type 2",
        "reason": "Neurological syndrome",
    },
    "1260327007": {
        "label":  "Familial hemiplegic migraine type 3",
        "reason": "Neurological syndrome",
    },
    "717003001": {
        "label":  "Hereditary cavernous hemangioma of brain",
        "reason": "Neurological — cerebral vascular malformation",
    },
    "56453003": {
        "label":  "Hereditary cerebral amyloid angiopathy, Dutch type",
        "reason": "Cerebrovascular — no cardiac diagnostic role",
    },
    "45639009": {
        "label":  "Hereditary cerebral amyloid angiopathy, Icelandic type",
        "reason": "Cerebrovascular — no cardiac diagnostic role",
    },
    "237867001": {
        "label":  "Hereditary cerebrovascular amyloidosis",
        "reason": "Cerebrovascular — no cardiac diagnostic role",
    },
    "1186724002": {
        "label":  "HtrA serine peptidase 1-related cerebral small vessel disease",
        "reason": "Neurological hereditary syndrome",
    },
    "724097003": {
        "label":  "Moyamoya angiopathy with short stature and facial dysmorphism",
        "reason": "Neurological syndrome",
    },
    "718551002": {
        "label":  "Moyamoya disease with early onset achalasia",
        "reason": "Neurological syndrome",
    },
    "1173997008": {
        "label":  "Pontine autosomal dominant microangiopathy",
        "reason": "Neurological hereditary syndrome",
    },
    "783787000": {
        "label":  "RVCL-S (retinal vasculopathy with cerebral leukoencephalopathy)",
        "reason": "Neurological hereditary syndrome",
    },
}


In [ ]:
# -------------------------------------------------------------------
# GROUP 9 — Hereditary dermatological syndromes
# Enter via vascular/skin shared ancestry in SNOMED.
# -------------------------------------------------------------------
HEREDITARY_DERMATOLOGICAL = {
    "36025004": {
        "label":  "Fibrous skin tumour of tuberous sclerosis",
        "reason": "Dermatology — enters via vascular malformation hierarchy",
    },
    "403776002": {
        "label":  "Hereditary cutaneous vascular syndrome",
        "reason": "Dermatology — vascular skin syndrome",
    },
    "403765001": {
        "label":  "Port-wine stain in Rubinstein-Taybi syndrome",
        "reason": "Dermatology — vascular skin malformation",
    },
    "763867001": {
        "label":  "SOLAMEN syndrome",
        "reason": "Dermatology — enters via vascular malformation hierarchy",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 10 — Hereditary renal / hepatic / skeletal syndromes
# -------------------------------------------------------------------
HEREDITARY_OTHER = {
    "367531000119106": {
        "label":  "Hereditary diffuse endocapillary proliferative glomerulonephritis",
        "reason": "Renal — no cardiac diagnostic role",
    },
    "763778003": {
        "label":  "Larsen-like syndrome (B3GAT3-related)",
        "reason": "Skeletal hereditary syndrome",
    },
    "733454004": {
        "label":  "Long thumb brachydactyly syndrome",
        "reason": "Skeletal syndrome",
    },
    "402851000": {
        "label":  "Neonatal purpura fulminans (protein C deficiency)",
        "reason": "Haematological neonatal complication",
    },
    "764100007": {
        "label":  "Primary intraosseous venous malformation",
        "reason": "Skeletal vascular malformation",
    },
}


In [ ]:
# -------------------------------------------------------------------
# GROUP 11 — Cardiovascular injury / neonatal failures
# -------------------------------------------------------------------
CARDIOVASCULAR_INJURY = {
    "284190000": {
        "label":  "Burn of cardiovascular structure",
        "reason": "Trauma concept — not a triage diagnostic entity",
    },
    "230690007": {
        "label":  "Cerebrovascular accident (stroke)",
        "reason": "Neurology scope — stroke managed separately from cardiac triage",
    },
    "95614004": {
        "label":  "Neonatal circulatory failure",
        "reason": "Neonatal medicine scope",
    },
}

In [ ]:
# -------------------------------------------------------------------
# GROUP 12 — Other individually confirmed contaminants
# Validated one by one during v1–v3 extraction audits.
# -------------------------------------------------------------------
CONFIRMED_CONTAMINANTS = {
    "56975005": {
        "label":  "Strawberry nevus of skin",
        "reason": "Dermatology — inherits via vascular lesion hierarchy",
    },
    "75971007": {
        "label":  "Choroidal retinal neovascularisation",
        "reason": "Ophthalmology contamination",
    },
    "10087007": {
        "label":  "Infection caused by Schistosoma",
        "reason": "Parasitology — enters via pulmonary HTN mechanism pathway",
    },
    "2043009": {
        "label":  "Alcoholic gastritis",
        "reason": "Gastroenterology contamination",
    },
    "408335007": {
        "label":  "Autoimmune hepatitis",
        "reason": "Hepatology contamination",
    },
    "193570009": {
        "label":  "Cataract",
        "reason": "Ophthalmology contamination",
    },
    "387800004": {
        "label":  "Cervical spondylosis",
        "reason": "Musculoskeletal contamination",
    },
    "196735001": {
        "label":  "Chronic superficial gastritis",
        "reason": "Gastroenterology contamination",
    },
    "846621001": {
        "label":  "Degenerative myopia left eye",
        "reason": "Ophthalmology contamination",
    },
    "846622008": {
        "label":  "Degenerative myopia right eye",
        "reason": "Ophthalmology contamination",
    },
    "128333008": {
        "label":  "Diarrheal disorder",
        "reason": "Gastroenterology contamination",
    },
    "257277002": {
        "label":  "Combined muscle and peripheral nerve disorder",
        "reason": "Neuromuscular contamination",
    },
    "1157107003": {
        "label":  "COVID vaccine procedure",
        "reason": "Procedure contamination — no cardiology relevance",
    },
    "68504005": {
        "label":  "Ataxia-telangiectasia syndrome",
        "reason": "Neurological/immunological syndrome",
    },
    "1351648007": {
        "label":  "Autosomal recessive agammaglobulinaemia",
        "reason": "Immunological disorder",
    },
    "720748007": {
        "label":  "Aural atresia with multiple congenital anomalies",
        "reason": "ENT/congenital syndrome",
    },
    "396275006": {
        "label":  "Osteoarthritis",
        "reason": "Musculoskeletal contamination",
    },
    "46775006": {
        "label":  "Neonatal respiratory distress syndrome",
        "reason": "Neonatal pulmonology — not a cardiac diagnostic finding",
    },
    "29061000087103": {
        "label":  "COVID vaccine product",
        "reason": "Vaccine product — no cardiology relevance",
    },
}



In [ ]:
# Group 13 — Noisy finding sites (non-cardiac body structures)
NOISY_FINDING_SITES = {
    "76752008":  {"label": "Breast structure",                        "reason": "Non-cardiac anatomy — finding site noise"},
    "80248007":  {"label": "Left breast structure",                   "reason": "Non-cardiac anatomy — finding site noise"},
    "73056007":  {"label": "Right breast structure",                  "reason": "Non-cardiac anatomy — finding site noise"},
    "59380008":  {"label": "Anterior abdominal wall structure",       "reason": "Non-cardiac anatomy — finding site noise"},
    "371398005": {"label": "Eye region structure",                    "reason": "Non-cardiac anatomy — finding site noise"},
    "80243003":  {"label": "Eyelid",                                  "reason": "Non-cardiac anatomy — finding site noise"},
    "81105003":  {"label": "Cervical lymph node",                     "reason": "Non-cardiac anatomy — finding site noise"},
    "89890002":  {"label": "Lymphatic system",                        "reason": "Non-cardiac anatomy — finding site noise"},
    "181768009": {"label": "Lymphatic tissue",                        "reason": "Non-cardiac anatomy — finding site noise"},
    "127856007": {"label": "Skin and/or subcutaneous tissue structure",  "reason": "Non-cardiac anatomy — finding site noise (2 terms, 1 concept)"},
    "76015000":  {"label": "Hepatic artery",                          "reason": "Non-cardiac anatomy — finding site noise"},
    "8993003":   {"label": "Hepatic vein",                            "reason": "Non-cardiac anatomy — finding site noise"},
    "34154005":  {"label": "Intrahepatic part of main portal vein",   "reason": "Non-cardiac anatomy — finding site noise"},
    "23451007":  {"label": "Adrenal structure",                       "reason": "Non-cardiac anatomy — finding site noise"},
    "2841007":   {"label": "Renal artery",                            "reason": "Non-cardiac anatomy — finding site noise"},
    "56400007":  {"label": "Renal vein",                              "reason": "Non-cardiac anatomy — finding site noise"},
}

In [ ]:
# ===================================================================
# REGISTER ALL EXCLUSION GROUPS
# The validation function and Cell 2 iterate this dict.
# ===================================================================

EXCLUSION_GROUPS = {
    "THERAPEUTIC_PROCEDURES":   THERAPEUTIC_PROCEDURES,
    "NON_CARDIO_ANATOMY":       NON_CARDIO_ANATOMY,
    "DERMATOLOGY_SKIN":         DERMATOLOGY_SKIN,
    "RETINAL_DIABETIC":         RETINAL_DIABETIC,
    "CKD_DIABETES_CROSS":       CKD_DIABETES_CROSS,
    "OBSTETRIC_NUTRITIONAL":    OBSTETRIC_NUTRITIONAL,
    "ROOT_LEVEL_CUTS":          ROOT_LEVEL_CUTS,
    "HEREDITARY_CEREBROVASCULAR": HEREDITARY_CEREBROVASCULAR,
    "HEREDITARY_DERMATOLOGICAL":  HEREDITARY_DERMATOLOGICAL,
    "HEREDITARY_OTHER":           HEREDITARY_OTHER,
    "CARDIOVASCULAR_INJURY":    CARDIOVASCULAR_INJURY,
    "CONFIRMED_CONTAMINANTS":   CONFIRMED_CONTAMINANTS,
    "NOISY_FINDING_SITES": NOISY_FINDING_SITES,
}

# Flat set used by Cell 2 BFS exclusion logic
EXCLUDED_SUBTREES = set()
for _group in EXCLUSION_GROUPS.values():
    EXCLUDED_SUBTREES |= set(_group.keys())

In [ ]:
# ===================================================================
# CONCEPT TYPE OVERRIDES
# Manually override the type label assigned by BFS for specific SCTIDs.
# ===================================================================

CONCEPT_TYPE_OVERRIDES = {
    "73211009":  "External",            # Diabetes mellitus (generic)
    "44054006":  "External",            # Diabetes mellitus type 2
    "46635009":  "External",            # Diabetes mellitus type 1
    "399208008": "Cardiology",          # Chest X-ray
}

In [ ]:
# ===================================================================
# VALIDATION
# ===================================================================

def validate_configuration():

    print("=" * 60)
    print(f"SNOMED CT Extraction Config — {CURATION_VERSION}  ({CURATION_DATE})")
    print("=" * 60)

    errors   = []
    warnings = []

    included = set(CARDIOLOGY_ROOTS.keys()) | set(PROTECTED_SEEDS.keys())
    excluded = EXCLUDED_ROOTS | EXCLUDED_SUBTREES

    # ------------------------------------------------------------------
    # 1. Overlap between included and excluded
    # ------------------------------------------------------------------
    overlap = included & excluded
    if overlap:
        errors.append(
            f"SCTIDs appear in BOTH inclusion and exclusion lists: {overlap}"
        )

    # ------------------------------------------------------------------
    # 2. No root is inside EXCLUDED_SUBTREES
    # ------------------------------------------------------------------
    for sctid in CARDIOLOGY_ROOTS:
        if sctid in EXCLUDED_SUBTREES:
            errors.append(f"Root {sctid} is in EXCLUDED_SUBTREES — will never be reached.")

    # ------------------------------------------------------------------
    # 3. No protected seed is inside EXCLUDED_SUBTREES
    # ------------------------------------------------------------------
    for sctid in PROTECTED_SEEDS:
        if sctid in EXCLUDED_SUBTREES:
            errors.append(f"Protected seed {sctid} is in EXCLUDED_SUBTREES — contradictory.")

    # ------------------------------------------------------------------
    # 4. Check for duplicates across exclusion groups
    # ------------------------------------------------------------------
    seen       = {}
    duplicates = []
    for group_name, group_data in EXCLUSION_GROUPS.items():
        for sctid in group_data:
            if sctid in seen:
                duplicates.append((sctid, seen[sctid], group_name))
            else:
                seen[sctid] = group_name

    if duplicates:
        for sctid, g1, g2 in duplicates:
            warnings.append(f"Duplicate: {sctid} in both '{g1}' and '{g2}' — harmless but tidy up.")

    # ------------------------------------------------------------------
    # 5. PROTECTED_SEEDS values must be plain strings (Cell 2 compat)
    # ------------------------------------------------------------------
    for sctid, val in PROTECTED_SEEDS.items():
        if not isinstance(val, str):
            errors.append(
                f"PROTECTED_SEEDS['{sctid}'] is {type(val).__name__}, expected str. "
                f"Cell 2 reads this as a plain type label — must be a string like 'Cardiology'."
            )

    # ------------------------------------------------------------------
    # Report
    # ------------------------------------------------------------------
    if warnings:
        print("\n⚠️  WARNINGS")
        for w in warnings:
            print(f"   {w}")

    if errors:
        print("\n❌ ERRORS — fix these before running extraction:")
        for e in errors:
            print(f"   {e}")
        raise ValueError(f"{len(errors)} configuration error(s) detected. See output above.")

    # ------------------------------------------------------------------
    # Summary table
    # ------------------------------------------------------------------
    print("\n📋 Root categories")
    print("-" * 40)
    counts = Counter(CARDIOLOGY_ROOTS.values())
    for label, n in sorted(counts.items()):
        print(f"   {label:<25} {n:>3} root(s)")

    print("\n📋 Exclusion groups")
    print("-" * 40)
    for group_name, group_data in EXCLUSION_GROUPS.items():
        print(f"   {group_name:<32} {len(group_data):>3} subtree(s)")

    print("\n📋 Totals")
    print("-" * 40)
    print(f"   Primary roots          : {len(CARDIOLOGY_ROOTS)}")
    print(f"   Protected seeds        : {len(PROTECTED_SEEDS)}")
    print(f"   Excluded top roots     : {len(EXCLUDED_ROOTS)}")
    print(f"   Excluded subtrees      : {len(EXCLUDED_SUBTREES)}  (across {len(EXCLUSION_GROUPS)} groups)")
    print(f"   Type overrides         : {len(CONCEPT_TYPE_OVERRIDES)}")

    print("\n✅ Configuration validated — no errors.")
    print("=" * 60)


validate_configuration()

SNOMED CT Extraction Config — v5.0  (2026-05-16)

📋 Root categories
----------------------------------------
   Cardiology                 13 root(s)

📋 Exclusion groups
----------------------------------------
   THERAPEUTIC_PROCEDURES            10 subtree(s)
   NON_CARDIO_ANATOMY                 5 subtree(s)
   DERMATOLOGY_SKIN                   2 subtree(s)
   RETINAL_DIABETIC                   5 subtree(s)
   CKD_DIABETES_CROSS                 6 subtree(s)
   OBSTETRIC_NUTRITIONAL              6 subtree(s)
   ROOT_LEVEL_CUTS                   13 subtree(s)
   HEREDITARY_CEREBROVASCULAR        17 subtree(s)
   HEREDITARY_DERMATOLOGICAL          4 subtree(s)
   HEREDITARY_OTHER                   5 subtree(s)
   CARDIOVASCULAR_INJURY              3 subtree(s)
   CONFIRMED_CONTAMINANTS            19 subtree(s)
   NOISY_FINDING_SITES               16 subtree(s)

📋 Totals
----------------------------------------
   Primary roots          : 13
   Protected seeds        : 6
   Excluded to

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load relationship file                                         ║
# ║  Loads ALL active SNOMED relationships into memory.      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def load_relationships():
    term_path = os.path.join(SNAPSHOT_PATH, "Terminology")
    fname = [f for f in os.listdir(term_path)
             if "Relationship_Snapshot" in f and "Stated" not in f][0]
    path = os.path.join(term_path, fname)
    print(f"  Loading: {fname}")
    chunks = []
    for chunk in pd.read_csv(path, sep='\t', dtype=str,
                             chunksize=BATCH_SIZE, low_memory=False):
        chunks.append(chunk[chunk['active'] == '1'])
    df = pd.concat(chunks, ignore_index=True)
    return df

os.makedirs(OUTPUT_DIR, exist_ok=True)
rels_df = load_relationships()
print(f"✅ CELL 2 complete — {len(rels_df):,} active relationships loaded.")

  Loading: sct2_Relationship_Snapshot_INT_20260301.txt
✅ CELL 2 complete — 1,331,550 active relationships loaded.


In [ ]:
import pandas as pd
from collections import deque

IS_A_TYPE = '116680003'

# Build child_map vectorized — takes seconds, not minutes
is_a_rels = rels_df[rels_df['typeId'] == IS_A_TYPE][['sourceId','destinationId']]

child_map = (
    is_a_rels.groupby('destinationId')['sourceId']
    .apply(list)
    .to_dict()
)

print(f"✅ child_map built — {len(child_map):,} parent nodes indexed")

# Optimized BFS — replace your existing bfs_descendants function
def bfs_descendants(root_ids, rels_df=None):
    """
    Uses pre-built child_map — rels_df parameter kept for compatibility
    but ignored. child_map must exist in scope.
    """
    visited = set(str(r) for r in root_ids)
    queue   = deque(str(r) for r in root_ids)

    while queue:
        current = queue.popleft()
        for child in child_map.get(current, []):  # O(1) lookup
            if child not in visited:
                visited.add(child)
                queue.append(child)

    return visited

print("✅ Optimized bfs_descendants ready")

# Initial BFS to get all cardiology IDs from CARDIOLOGY_ROOTS
# This step was missing and caused `all_cardio_ids` to be undefined.
all_cardio_ids = bfs_descendants(CARDIOLOGY_ROOTS.keys())
print(f"✅ Initial all_cardio_ids built from roots: {len(all_cardio_ids):,} concepts")

✅ child_map built — 133,230 parent nodes indexed
✅ Optimized bfs_descendants ready
✅ Initial all_cardio_ids built from roots: 8,404 concepts


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Apply exclusions, then restore protected seeds                 ║
# ║  Key fix: protected seeds are added AFTER exclusions so they survive.    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# BFS from exclusion roots to get everything to remove
excluded_ids = bfs_descendants(EXCLUDED_SUBTREES, rels_df)
excluded_ids.update(bfs_descendants(EXCLUDED_ROOTS, rels_df))

before = len(all_cardio_ids)
all_cardio_ids -= excluded_ids
after  = len(all_cardio_ids)

print(f"  Excluded IDs : {len(excluded_ids):,}")
print(f"  Before       : {before:,}  →  After: {after:,}  (removed {before - after:,})")

# ── Restore protected seeds ───────────────────────────────────────────────────
# Some protected seeds may be descendants of excluded subtrees.
# We add them back explicitly so they always survive.
# PROTECTED_SEEDS values are plain strings — used directly as conceptType.
restored = []
for sctid, ctype in PROTECTED_SEEDS.items():
    if sctid not in all_cardio_ids:
        all_cardio_ids.add(sctid)
        restored.append(sctid)

if restored:
    print(f"\n  ℹ️  Restored {len(restored)} protected seed(s) removed by exclusion:")
    for s in restored:
        print(f"     {s}  ({PROTECTED_SEEDS[s]})")
else:
    print("\n  ✅ All protected seeds survived exclusion naturally.")

# ── Spot checks ───────────────────────────────────────────────────────────────
spot_checks = {
    '16652001':  "Fabry disease         [protected seed]",
    '31848007':  "CREST syndrome        [protected seed]",
    '401314000': "NSTEMI                [protected seed]",
    '301120008': "ECG finding root      [procedure root]",
    '230145002': "Dyspnea               [symptom root]",
    '390917008': "BNP measurement       [lab root]",
    '25607008': "D-dimer measurement   [lab root]",
}
print()
for sctid, label in spot_checks.items():
    status = '✅' if sctid in all_cardio_ids else '❌ MISSING'
    print(f"  {status}  {sctid}  {label}")

print(f"\n✅ CELL 4 complete — {len(all_cardio_ids):,} concepts after exclusions + restoration.")

  Excluded IDs : 65,070
  Before       : 8,404  →  After: 7,200  (removed 1,204)

  ℹ️  Restored 2 protected seed(s) removed by exclusion:
     16652001  (Cardiology)
     13645005  (External)

  ✅  16652001  Fabry disease         [protected seed]
  ✅  31848007  CREST syndrome        [protected seed]
  ✅  401314000  NSTEMI                [protected seed]
  ✅  301120008  ECG finding root      [procedure root]
  ✅  230145002  Dyspnea               [symptom root]
  ✅  390917008  BNP measurement       [lab root]
  ✅  25607008  D-dimer measurement   [lab root]

✅ CELL 4 complete — 7,202 concepts after exclusions + restoration.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Collect External concepts                                      ║
# ║  Any concept referenced by a cardiology concept via any relationship     ║
# ║  (FINDING_SITE, ASSOCIATED_MORPHOLOGY, etc.) but not itself cardiology.  ║
# ║  NOISY_FINDING_SITES are already in EXCLUDED_SUBTREES → excluded_ids,    ║
# ║  so the line `external_ids -= excluded_ids` removes them automatically.  ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

external_ids = set()
for _, row in rels_df.iterrows():
    src = str(row['sourceId'])
    dst = str(row['destinationId'])
    if src in all_cardio_ids and dst not in all_cardio_ids:
        external_ids.add(dst)

# Remove anything that belongs to an excluded branch (incl. noisy finding sites)
external_ids -= excluded_ids
external_ids -= all_cardio_ids   # safety: no overlap

all_ids = all_cardio_ids | external_ids

print(f"✅ CELL 5 complete — External concepts collected.")
print(f"   Cardiology : {len(all_cardio_ids):,}")
print(f"   External   : {len(external_ids):,}")
print(f"   Total      : {len(all_ids):,}")

✅ CELL 5 complete — External concepts collected.
   Cardiology : 7,202
   External   : 3,033
   Total      : 10,235


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Build concept type map & apply overrides                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# Base assignment
concept_type_map = {}
for sctid in all_cardio_ids:
    # Roots keep their declared type; everything else is 'Cardiology'
    concept_type_map[sctid] = CARDIOLOGY_ROOTS.get(sctid, 'Cardiology')
for sctid in external_ids:
    concept_type_map[sctid] = 'External'

# Apply CONCEPT_TYPE_OVERRIDES (from Cell 1)
for sctid, ctype in CONCEPT_TYPE_OVERRIDES.items():
    if sctid in concept_type_map:
        concept_type_map[sctid] = ctype

# Apply PROTECTED_SEEDS types — must come after overrides so seeds win
for sctid, ctype in PROTECTED_SEEDS.items():
    if sctid in concept_type_map:
        concept_type_map[sctid] = ctype

# ── Verify overrides applied ──────────────────────────────────────────────────
print("  Override verification:")
for sctid, expected in CONCEPT_TYPE_OVERRIDES.items():
    actual = concept_type_map.get(sctid, 'NOT IN GRAPH')
    status = '✅' if actual == expected else '⚠️ '
    print(f"   {status} {sctid} → {actual}  (expected {expected})")

print("\n  Protected seed types:")
for sctid, expected in PROTECTED_SEEDS.items():
    actual = concept_type_map.get(sctid, 'NOT IN GRAPH')
    status = '✅' if actual == expected else '⚠️ '
    print(f"   {status} {sctid} → {actual}")

print(f"\n✅ CELL 6 complete — type map built ({len(concept_type_map):,} entries).")

  Override verification:
   ✅ 73211009 → External  (expected External)
   ✅ 44054006 → External  (expected External)
   ✅ 46635009 → External  (expected External)
   ✅ 399208008 → DiagnosticProcedure  (expected DiagnosticProcedure)

  Protected seed types:
   ✅ 16652001 → Cardiology
   ✅ 31848007 → Cardiology
   ✅ 13645005 → External
   ✅ 398254007 → External
   ✅ 401314000 → Cardiology
   ✅ 225566008 → Cardiology

✅ CELL 6 complete — type map built (10,235 entries).


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Load & filter Concept file                                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

term_path = os.path.join(SNAPSHOT_PATH, "Terminology")

concepts_file = [f for f in os.listdir(term_path) if "Concept_Snapshot" in f][0]
print(f"  Loading: {concepts_file}")

concepts_raw = pd.read_csv(os.path.join(term_path, concepts_file),
                           sep='\t', dtype=str)
concepts = concepts_raw[
    (concepts_raw['active'] == '1') &
    (concepts_raw['id'].isin(all_ids))
].copy()

concepts['conceptType'] = concepts['id'].map(concept_type_map)
concepts['definitionStatus'] = concepts['definitionStatusId'].map({
    '900000000000074008': 'Primitive',
    '900000000000073002': 'FullyDefined',
}).fillna('Primitive')

concepts = concepts[['id', 'conceptType', 'definitionStatus']].rename(columns={'id': 'sctid'})
concepts = concepts.drop_duplicates(subset=['sctid'])

print(f"✅ CELL 7 complete — {len(concepts):,} concept nodes ready.")

  Loading: sct2_Concept_Snapshot_INT_20260301.txt
✅ CELL 7 complete — 10,235 concept nodes ready.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Load & filter Description file                                 ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

desc_file = [f for f in os.listdir(term_path)
             if "Description_Snapshot-en" in f][0]
print(f"  Loading: {desc_file}")

descriptions = pd.read_csv(os.path.join(term_path, desc_file),
                           sep='\t', dtype=str)
descriptions = descriptions[
    (descriptions['active'] == '1') &
    (descriptions['conceptId'].isin(all_ids))
].copy()
descriptions = descriptions.drop_duplicates(subset=['conceptId', 'term'])

print(f"✅ CELL 8 complete — {len(descriptions):,} descriptions ready.")

  Loading: sct2_Description_Snapshot-en_INT_20260301.txt
✅ CELL 8 complete — 29,180 descriptions ready.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Filter relationships & apply TYPE_MAP                          ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

TYPE_MAP = {
    '116680003': 'IS_A',
    '363698007': 'FINDING_SITE',
    '116676008': 'ASSOCIATED_MORPHOLOGY',
    '246454002': 'OCCURRENCE',
    '370135005': 'PATHOLOGICAL_PROCESS',
    '42752001':  'DUE_TO',
    '363713009': 'HAS_INTERPRETATION',
    '363714003': 'INTERPRETS',
    '255234002': 'AFTER',
    '246090004': 'ASSOCIATED_FINDING',
    '408729009': 'FINDING_CONTEXT',
    '408730004': 'PROCEDURE_CONTEXT',
    '408731000': 'TEMPORAL_CONTEXT',
    '408732007': 'SUBJECT_RELATIONSHIP_CONTEXT',
    '47429007':  'ASSOCIATED_WITH',
    '363701004': 'DIRECT_MORPHOLOGY',
    '405813007': 'PROCEDURE_SITE_DIRECT',
    '405816004': 'PROCEDURE_MORPHOLOGY',
    '260686004': 'METHOD',
    '424244007': 'USING_ENERGY',
    '260507000': 'ACCESS',
    '363702006': 'HAS_FOCUS',
    '363710007': 'INDIRECT_DEVICE',
    '405815000': 'PROCEDURE_DEVICE',
    '363709002': 'INDIRECT_MORPHOLOGY',
    '246513007': 'REVISION_STATUS',
    '308489006': 'PATHOLOGICAL_PROCESS',
    '118171006': 'SPECIMEN_PROCEDURE',
    '370132008': 'SCALE_TYPE',
    '370130000': 'PROPERTY',
    '704327008': 'DIRECT_SITE',
    '704326004': 'PRECONDITION',
    '263502005': 'CLINICAL_COURSE',
    '246075003': 'CAUSATIVE_AGENT',
    '363703001': 'HAS_INTENT',
    '405814001': 'PROCEDURE_SITE_INDIRECT',
    '363704007': 'PROCEDURE_SITE',
    '116688005': 'PROCEDURE_APPROACH',
    '371881003': 'DURING',
    '424361007': 'USING_SUBSTANCE',
    '363700003': 'DIRECT_MORPHOLOGY',
    '726633004': 'TEMPORALLY_RELATED_TO',
    '246093002': 'COMPONENT',
    '419066007': 'FINDING_INFORMER',
    '116686009': 'HAS_SPECIMEN',
    '704319004': 'INHERENT_IN',
}

rels_filtered = rels_df[
    (rels_df['sourceId'].isin(all_ids)) &
    (rels_df['destinationId'].isin(all_ids))
].copy()

rels_filtered = rels_filtered.drop_duplicates(
    subset=['sourceId', 'typeId', 'destinationId']
)
rels_filtered['relType'] = rels_filtered['typeId'].map(TYPE_MAP).fillna(
    rels_filtered['typeId'].apply(lambda x: f'SNOMED_{x}')
)

unmapped = rels_filtered[rels_filtered['relType'].str.startswith('SNOMED_')]
if len(unmapped) > 0:
    print(f"  ⚠️  {len(unmapped):,} unmapped typeIds → SNOMED_xxx fallback:")
    print(unmapped['typeId'].value_counts().head(10).to_string())
else:
    print("  ✅ All relationship types mapped cleanly.")

print(f"✅ CELL 9 complete — {len(rels_filtered):,} relationships ready.")

  ✅ All relationship types mapped cleanly.
✅ CELL 9 complete — 44,165 relationships ready.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Write output TSV files                                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

concepts.to_csv(os.path.join(OUTPUT_DIR, "cardio_concepts.tsv"),
                sep='\t', index=False)
descriptions.to_csv(os.path.join(OUTPUT_DIR, "cardio_descriptions.tsv"),
                    sep='\t', index=False)
rels_filtered.to_csv(os.path.join(OUTPUT_DIR, "cardio_relationships.tsv"),
                     sep='\t', index=False)

counts = {
    'concepts':      len(concepts),
    'descriptions':  len(descriptions),
    'relationships': len(rels_filtered),
}

print("✅ CELL 10 complete — TSV files written.")
print(f"   cardio_concepts.tsv      : {counts['concepts']:>8,}")
print(f"   cardio_descriptions.tsv  : {counts['descriptions']:>8,}")
print(f"   cardio_relationships.tsv : {counts['relationships']:>8,}")

✅ CELL 10 complete — TSV files written.
   cardio_concepts.tsv      :   10,235
   cardio_descriptions.tsv  :   29,180
   cardio_relationships.tsv :   44,165


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Validation gate                                                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

VALIDATION_CONTRACT = {
    "min_concepts":           9000,
    "max_concepts":          45000,
    "min_descriptions":      20000,
    "min_relationships":     40000,
    "max_isa_percentage":       50,
    "min_rel_types":              8,
    "min_cardiology_concepts": 7000,
    "max_external_concepts":   6000,

    "required_sctids": [
        # Core diseases (descendants of 49601007 — must always be present)
        '22298006',   # Myocardial infarction
        '84114007',   # Heart failure
        '49436004',   # Atrial fibrillation
        '38341003',   # Hypertensive disorder
        '57054005',   # Acute myocardial infarction

        # Symptoms (roots in SYMPTOM_ROOTS)
        '29857009',   # Chest pain
        '230145002',  # Dyspnea
        '80313002',   # Palpitations
        '271594007',  # Syncope
        '52613005',   # Diaphoresis

        # Diagnostic finding root
        '301120008',  # Finding present on electrocardiogram

        # Procedure roots
        '40701008',   # Echocardiography
        '29303009',   # ECG procedure

        # Lab roots — use verified SCTIDs from Cell 1
        '105000003',  # Troponin measurement
        '390917008',  # BNP measurement
        '25607008',  # D-dimer measurement

        # Type overrides — must be External in graph
        '73211009',   # Diabetes mellitus

        # Protected seeds — must survive exclusion
        '16652001',   # Fabry disease
        '31848007',   # CREST syndrome
        '398254007',  # Pre-eclampsia
        '13645005',   # COPD
        '401314000',  # NSTEMI
        '225566008',  # Ischaemic chest pain
    ],

    "forbidden_sctids": [
        # ── Therapeutic procedures (Group 1 exclusions) ───────────────────
        '415070008',        # PCI
        '232717009',        # CABG
        '18286008',         # Cardiac ablation
        '233174007',        # Pacemaker implantation
        '395218007',        # ICD insertion (Implantation of internal cardiac defibrillator)
        '64915003',         # Valve operation
        '32413006',         # Heart transplant
        '232965003',        # VAD insertion
        '704050007',        # Cardiac rehab

        # ── Noisy finding sites (Group 13) ────────────────────────────────
        '76752008',         # Breast structure
        '80248007',         # Left breast structure
        '73056007',         # Right breast structure
        '59380008',         # Anterior abdominal wall
        '371398005',        # Eye region structure
        '80243003',         # Eyelid
        '81105003',         # Cervical lymph node
        '89890002',         # Lymphatic system
        '181768009',        # Lymphatic tissue
        '127856007',        # Skin and subcutaneous tissue
        '76015000',         # Hepatic artery
        '8993003',          # Hepatic vein
        '34154005',         # Intrahepatic part of main portal vein
        '23451007',         # Adrenal structure
        '2841007',          # Renal artery
        '56400007',         # Renal vein

        # ── Original confirmed contaminants ───────────────────────────────
        '56975005',         # Strawberry nevus
        '75971007',         # Choroidal retinal neovascularisation
        '10087007',         # Schistosoma infection
        '193570009',        # Cataract
        '846621001',        # Degenerative myopia left eye
        '846622008',        # Degenerative myopia right eye
        '387800004',        # Cervical spondylosis
        '2043009',          # Alcoholic gastritis
        '95614004',         # Neonatal circulatory failure
        '75524006',         # Malnutrition-related diabetes
        '276792008',        # Pulm HTN with extreme obesity
        '1264255000',       # Induced termination of pregnancy
        '25412000',         # Retinal microaneurysm due to DM
        '314014002',        # Ischaemic maculopathy due to DM
        '140111000119107',  # HTN in CKD stage 4 due to T2DM
        '1332436006',       # HTN in CKD stage 3B due to T1DM

        # ── Root-level cuts ───────────────────────────────────────────────
        '62914000',         # Cerebrovascular disease
        '432995009',        # Fetal cardiovascular disorder
        '363696006',        # Neonatal cardiovascular disorder
        '276510003',        # Perinatal cardiovascular disorders
        '397893009',        # Cardiovascular morbidity
        '721573003',        # Neoplasm of cardiovascular system

        # ── Hereditary cuts ───────────────────────────────────────────────
        '771476007',        # Autosomal recessive leukoencephalopathy
        '703219008',        # CARASIL
        '95656000',         # Familial hemiplegic migraine
        '717003001',        # Hereditary cavernous hemangioma of brain
        '230690007',        # Cerebrovascular accident
        '284190000',        # Burn of cardiovascular structure
        '68504005',         # Ataxia-telangiectasia
        '36025004',         # Fibrous skin tumour of tuberous sclerosis
        '403765001',        # Port-wine stain in Rubinstein-Taybi
        '402851000',        # Neonatal purpura fulminans
    ],
}

# ── Run checks ────────────────────────────────────────────────────────────────
failures = []
warnings = []
passed   = []

def ok(m):   passed.append(f"  ✅  {m}")
def fail(m): failures.append(f"  ❌  {m}")
def warn(m): warnings.append(f"  ⚠️   {m}")

c = counts['concepts']
if   c < VALIDATION_CONTRACT['min_concepts']: fail(f"Too few concepts: {c:,}")
elif c > VALIDATION_CONTRACT['max_concepts']: fail(f"Too many concepts: {c:,}")
else: ok(f"Concept count OK: {c:,}")

d = counts['descriptions']
if d < VALIDATION_CONTRACT['min_descriptions']: fail(f"Too few descriptions: {d:,}")
else: ok(f"Description count OK: {d:,}")

r = counts['relationships']
if r < VALIDATION_CONTRACT['min_relationships']: fail(f"Too few relationships: {r:,}")
else: ok(f"Relationship count OK: {r:,}")

type_counts = concepts['conceptType'].value_counts().to_dict()
cardio_n    = sum(v for k, v in type_counts.items() if k != 'External')
extern_n    = type_counts.get('External', 0)

if cardio_n < VALIDATION_CONTRACT['min_cardiology_concepts']:
    fail(f"Too few Cardiology concepts: {cardio_n:,}")
else:
    ok(f"Cardiology count OK: {cardio_n:,}")

if extern_n > VALIDATION_CONTRACT['max_external_concepts']:
    warn(f"Many External concepts: {extern_n:,}  (limit {VALIDATION_CONTRACT['max_external_concepts']:,})")
else:
    ok(f"External count OK: {extern_n:,}")

present         = set(concepts['sctid'].astype(str))
missing_req     = [s for s in VALIDATION_CONTRACT['required_sctids'] if s not in present]
found_forbidden = [s for s in VALIDATION_CONTRACT['forbidden_sctids'] if s in present]

if missing_req:
    fail(f"Missing required SCTIDs: {missing_req}")
else:
    ok(f"All {len(VALIDATION_CONTRACT['required_sctids'])} required SCTIDs present")

if found_forbidden:
    fail(f"Forbidden SCTIDs still present: {found_forbidden}")
else:
    ok(f"All {len(VALIDATION_CONTRACT['forbidden_sctids'])} forbidden SCTIDs absent")

rel_types = rels_filtered['typeId'].value_counts()
total_r   = len(rels_filtered)
isa_pct   = round(100 * rel_types.get(IS_A_TYPE, 0) / total_r, 1) if total_r else 100

if isa_pct > VALIDATION_CONTRACT['max_isa_percentage']:
    fail(f"IS_A is {isa_pct}% — too dominant (limit {VALIDATION_CONTRACT['max_isa_percentage']}%) ")
else:
    ok(f"IS_A percentage OK: {isa_pct}%")

n_types = len(rel_types)
if n_types < VALIDATION_CONTRACT['min_rel_types']:
    fail(f"Only {n_types} relationship types (need {VALIDATION_CONTRACT['min_rel_types']})")
else:
    ok(f"Relationship type diversity OK: {n_types} types")

concepts_with_desc = set(descriptions['conceptId'].astype(str))
orphans = present - concepts_with_desc
if orphans:
    fail(f"{len(orphans):,} orphan concepts (no description)")
else:
    ok("No orphan concepts")

dup_check = rels_filtered.duplicated(subset=['sourceId', 'typeId', 'destinationId'])
if dup_check.sum() > 0:
    fail(f"{dup_check.sum():,} duplicate relationships")
else:
    ok("No duplicate relationships")

# ── Print results ─────────────────────────────────────────────────────────────
print()
for m in passed:   print(m)
for m in warnings: print(m)
for m in failures: print(m)

report = {
    "timestamp": datetime.now().isoformat(),
    "curation_version": CURATION_VERSION,
    "counts": counts,
    "passed":   len(passed),
    "warnings": len(warnings),
    "failures": len(failures),
    "details":  {"passed": passed, "warnings": warnings, "failures": failures},
}
with open(os.path.join(OUTPUT_DIR, "validation_report.json"), 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n  Report saved → {OUTPUT_DIR}/validation_report.json")

if failures:
    print("\n" + "═" * 55)
    print(f"  ❌  FAILED — {len(failures)} check(s) did not pass.")
    print("  ⛔  DO NOT IMPORT TO NEO4J.")
    print("═" * 55)
    raise SystemExit(1)
else:
    print("\n" + "═" * 55)
    print(f"  ✅  ALL CHECKS PASSED — {len(passed)} checks, {len(warnings)} warnings.")
    print("  🟢  Safe to proceed to CELL 12.")
    print("═" * 55)



  ✅  Concept count OK: 10,235
  ✅  Description count OK: 29,180
  ✅  Relationship count OK: 44,165
  ✅  Cardiology count OK: 7,201
  ✅  External count OK: 3,034
  ✅  All 23 required SCTIDs present
  ✅  All 57 forbidden SCTIDs absent
  ✅  IS_A percentage OK: 43.1%
  ✅  Relationship type diversity OK: 28 types
  ✅  No orphan concepts
  ✅  No duplicate relationships

  Report saved → /content/drive/MyDrive/PFE/MedGraph/validation_report.json

═══════════════════════════════════════════════════════
  ✅  ALL CHECKS PASSED — 11 checks, 0 warnings.
  🟢  Safe to proceed to CELL 12.
═══════════════════════════════════════════════════════


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Prepare Neo4j CSV files                                       ║
# ║  termType column is now human-readable text: 'FSN' or 'Synonym'.         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# Map SNOMED typeId numbers to readable labels for the termType column
TERM_TYPE_MAP = {
    '900000000000003001': 'FSN',        # Fully Specified Name
    '900000000000013009': 'Synonym',    # Acceptable synonym
}

neo4j_dir = os.path.join(OUTPUT_DIR, "neo4j_import")
os.makedirs(neo4j_dir, exist_ok=True)

# ── nodes_concepts.csv ────────────────────────────────────────────────────────
concepts.to_csv(os.path.join(neo4j_dir, "nodes_concepts.csv"), index=False)

# ── nodes_terms.csv ───────────────────────────────────────────────────────────
terms = descriptions[['id', 'term']].rename(columns={'id': 'termId'})
terms = terms.drop_duplicates(subset=['termId'])
terms.to_csv(os.path.join(neo4j_dir, "nodes_terms.csv"), index=False)

# ── rels_has_term.csv — termType as readable text ─────────────────────────────
rels_has_term = descriptions[['conceptId', 'id', 'typeId']].copy()
rels_has_term.columns = ['startId', 'endId', 'termType']
rels_has_term['termType'] = (
    rels_has_term['termType']
    .map(TERM_TYPE_MAP)
    .fillna('Other')   # any undocumented typeId stays as 'Other'
)
rels_has_term = rels_has_term.drop_duplicates(subset=['startId', 'endId'])
rels_has_term.to_csv(os.path.join(neo4j_dir, "rels_has_term.csv"), index=False)

# ── rels_snomed.csv ───────────────────────────────────────────────────────────
rels_snomed = rels_filtered[['sourceId', 'destinationId', 'relType']].copy()
rels_snomed.columns = ['startId', 'endId', 'relType']
rels_snomed = rels_snomed.drop_duplicates(subset=['startId', 'endId', 'relType'])
rels_snomed.to_csv(os.path.join(neo4j_dir, "rels_snomed.csv"), index=False)

print("✅ CELL 12 complete — Neo4j import files written.")
print(f"   nodes_concepts.csv : {len(concepts):,}")
print(f"   nodes_terms.csv    : {len(terms):,}")
print(f"   rels_has_term.csv  : {len(rels_has_term):,}   (termType: FSN / Synonym / Other)")
print(f"   rels_snomed.csv    : {len(rels_snomed):,}")
print(f"\n   Folder: {neo4j_dir}")
print("   Copy this folder to your Neo4j import directory.")

✅ CELL 12 complete — Neo4j import files written.
   nodes_concepts.csv : 10,235
   nodes_terms.csv    : 29,180
   rels_has_term.csv  : 29,180   (termType: FSN / Synonym / Other)
   rels_snomed.csv    : 44,165

   Folder: /content/drive/MyDrive/PFE/MedGraph/neo4j_import
   Copy this folder to your Neo4j import directory.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL A — Reload saved CSV files for inspection               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

NEO4J_IMPORT_DIR = os.path.join(OUTPUT_DIR, "neo4j_import")

nodes_concepts = pd.read_csv(f"{NEO4J_IMPORT_DIR}/nodes_concepts.csv", dtype=str)
nodes_terms    = pd.read_csv(f"{NEO4J_IMPORT_DIR}/nodes_terms.csv",    dtype=str)
rels_has_term  = pd.read_csv(f"{NEO4J_IMPORT_DIR}/rels_has_term.csv",  dtype=str)
rels_snomed    = pd.read_csv(f"{NEO4J_IMPORT_DIR}/rels_snomed.csv",    dtype=str)

# Alias to match variable names used by Cell H search function
concepts  = nodes_concepts
terms     = nodes_terms

print("Files reloaded:")
print(f"  nodes_concepts : {len(nodes_concepts):,}")
print(f"  nodes_terms    : {len(nodes_terms):,}")
print(f"  rels_has_term  : {len(rels_has_term):,}")
print(f"  rels_snomed    : {len(rels_snomed):,}")

# Quick termType distribution check
print("\n  termType distribution in rels_has_term:")
print(rels_has_term['termType'].value_counts().to_string())

Files reloaded:
  nodes_concepts : 10,235
  nodes_terms    : 29,180
  rels_has_term  : 29,180
  rels_snomed    : 44,165

  termType distribution in rels_has_term:
termType
Synonym    18945
FSN        10235


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL H — Clinical concept search by NAME                                 ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

FSN_TYPE = 'FSN'       # after TERM_TYPE_MAP mapping in Cell 12
SYN_TYPE = 'Synonym'   # after TERM_TYPE_MAP mapping in Cell 12

# ── Build lookup tables ───────────────────────────────────────────────────────
type_lookup = concepts.set_index('sctid')['conceptType'].to_dict()
out_rels    = rels_snomed.groupby('startId').size().to_dict()
in_rels     = rels_snomed.groupby('endId').size().to_dict()

all_terms_merged = (
    rels_has_term
    .merge(terms, left_on='endId', right_on='termId', how='left')
)
fsn_terms = all_terms_merged[all_terms_merged['termType'] == FSN_TYPE]
syn_terms = all_terms_merged[all_terms_merged['termType'] == SYN_TYPE]

fsn_lookup = fsn_terms.set_index('startId')['term'].to_dict()
syn_lookup  = syn_terms.groupby('startId')['term'].apply(list).to_dict()


def search(keyword, max_results=5):
    kw = keyword.lower().strip()
    fsn_hits = fsn_terms[fsn_terms['term'].str.lower().str.contains(kw, na=False)]
    syn_hits = syn_terms[syn_terms['term'].str.lower().str.contains(kw, na=False)]

    fsn_sctids = set(fsn_hits['startId'].tolist())
    syn_sctids = set(syn_hits['startId'].tolist()) - fsn_sctids
    all_sctids = list(fsn_sctids) + list(syn_sctids)

    if not all_sctids:
        print(f"  ❌  NOT FOUND  '{keyword}'")
        return

    for sctid in all_sctids[:max_results]:
        fsn      = fsn_lookup.get(sctid, '— no FSN —')
        synonyms = syn_lookup.get(sctid, [])
        ctype    = type_lookup.get(sctid, '?')
        out_c    = out_rels.get(sctid, 0)
        in_c     = in_rels.get(sctid, 0)
        match_lbl = 'FSN' if sctid in fsn_sctids else 'syn'

        print(f"  ✅  {sctid:<15} [{ctype:<22}]  "
              f"out={out_c:>3}  in={in_c:>3}  syn={len(synonyms):>2}  "
              f"[{match_lbl}]  {fsn[:55]}")
        if synonyms:
            preview = ', '.join(f'"{s}"' for s in synonyms[:3])
            if len(synonyms) > 3:
                preview += f'  … +{len(synonyms) - 3} more'
            print(f"               ↳ {preview}")

    if len(all_sctids) > max_results:
        print(f"               … {len(all_sctids) - max_results} more — use a more specific keyword")


# ── Example searches — edit freely ───────────────────────────────────────────
print("═" * 70)
print("  SAMPLE SEARCHES — ACS pathway")
print("═" * 70)

print("\n── Symptoms ─────────────────────────────────────────────────────────")
search("chest pain")
search("dyspnea")
search("diaphoresis")
search("palpitation")
search("syncope")

print("\n── ACS diagnoses ────────────────────────────────────────────────────")
search("myocardial infarction")
search("STEMI")
search("NSTEMI")
search("acute coronary syndrome")
search("unstable angina")

print("\n── ECG findings ─────────────────────────────────────────────────────")
search("ST elevation")
search("ST depression")
search("left bundle branch block")
search("T wave inversion")

print("\n── Labs ─────────────────────────────────────────────────────────────")
search("troponin")
search("natriuretic peptide")
search("D-dimer")

print("\n── Heart failure ────────────────────────────────────────────────────")
search("heart failure")
search("ejection fraction")
search("oedema")

print("\n── Arrhythmia ───────────────────────────────────────────────────────")
search("atrial fibrillation")
search("ventricular tachycardia")
search("ventricular fibrillation")
search("heart block")

══════════════════════════════════════════════════════════════════════
  SAMPLE SEARCHES — ACS pathway
══════════════════════════════════════════════════════════════════════

── Symptoms ─────────────────────────────────────────────────────────
  ✅  102587001       [Cardiology            ]  out=  4  in=  1  syn= 1  [FSN]  Acute chest pain (finding)
               ↳ "Acute chest pain"
  ✅  281245003       [Cardiology            ]  out=  4  in=  8  syn= 1  [FSN]  Musculoskeletal chest pain (finding)
               ↳ "Musculoskeletal chest pain"
  ✅  279019008       [Cardiology            ]  out=  3  in=  0  syn= 1  [FSN]  Central crushing chest pain (finding)
               ↳ "Central crushing chest pain"
  ✅  36859004        [Cardiology            ]  out=  3  in=  0  syn= 2  [FSN]  Esophageal chest pain (finding)
               ↳ "Esophageal chest pain", "Oesophageal chest pain"
  ✅  285389008       [Cardiology            ]  out=  2  in=  0  syn= 1  [FSN]  Upper chest pain (finding)
   